# Retrieval metrics usage

## 1. What these retrieval metrics measure

The retrieval evaluators compare the query in `EvaluationCase.input` with the ranked list in `retrieved_documents`. Relevance@K measures top-K precision, Hit Rate@K detects whether retrieval found anything useful, MRR@K rewards an early first relevant result, and nDCG@K measures the ordering of all relevant results. Contextual Relevancy evaluates useful information inside retrieval, Contextual Precision@K evaluates AP-style ordering, and Contextual Recall compares retrieval with authoritative `context`. No generated `output` is required.

## 2. Imports

In [ ]:
from idp_eval import (
    ContextualPrecisionAtKEvaluator,
    ContextualRecallEvaluator,
    ContextualRelevancyEvaluator,
    EvaluationCase,
    EvaluationFramework,
    HitRateAtKEvaluator,
    MRRAtKEvaluator,
    NDCGAtKEvaluator,
    RelevanceAtKEvaluator,
    create_azure_judge,
)
from idp_eval.judges import AzureJudgeConfig

## 3. Configure the judge

Applications should inject configuration from their own settings/secrets layer. These values are placeholders only. `create_gateway_judge(config=...)` is also supported; retrieval semantics are backend-independent.

In [ ]:
azure_config = AzureJudgeConfig(
    model="your-azure-deployment",
    azure_endpoint="https://your-resource.openai.azure.com",
    tenant_id="your-tenant-id",
    client_id="your-client-id",
    client_secret="your-client-secret",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)
judge = create_azure_judge(config=azure_config)

## 4. Build a retrieval case

`input` is the retrieval query. `retrieved_documents` is the ranked retrieval result, and list order is rank. Retriever similarity `score` is diagnostic metadata only—it is never sent to the LLM relevance judge.

In [ ]:
case = EvaluationCase(
    case_id="retrieval-001",
    input="How do I reset my password?",
    retrieved_documents=[
        {
            "document_id": "doc-1",
            "text": "Open the account recovery page and select Reset Password.",
            "score": 0.95,
        },
        {
            "document_id": "doc-2",
            "text": "Billing invoices are available from the payments page.",
            "score": 0.91,
        },
        {
            "document_id": "doc-3",
            "text": "A password reset link is sent to the verified email address.",
            "score": 0.88,
        },
        {
            "document_id": "doc-4",
            "text": "Profile photos can be updated from account settings.",
            "score": 0.82,
        },
        {
            "document_id": "doc-5",
            "text": "If the reset email does not arrive, check spam or request another link.",
            "score": 0.79,
        },
    ],
)

## 5. Configure the four shared document metrics

In [ ]:
framework = EvaluationFramework(
    judge=judge,
    evaluators=[
        RelevanceAtKEvaluator(k=5),
        HitRateAtKEvaluator(k=5),
        MRRAtKEvaluator(k=5),
        NDCGAtKEvaluator(k=5),
    ],
)
framework.metrics

## 6. Run the shared document metrics

In [ ]:
results = framework.evaluate(case)
results["relevance_at_5"]
results["hit_rate_at_5"]
results["mrr_at_5"]
results["ndcg_at_5"]

## 7. Understand the shared relevance judgments

All four metrics share **one semantic LLM relevance judgment**:

```text
query + ranked top-K document text
              ↓
one relevance judge call
              ↓
binary per-document judgments
              ↓
       [1, 0, 1, 0, 1]
              ↓
Python computes all four metrics
```

The LLM returns only rank, a binary relevance judgment, and a reason. It does **not** calculate Relevance@K, Hit Rate@K, MRR@K, DCG, IDCG, or nDCG@K.

## 8. Relevance@K

`Relevance@K = relevant documents / effective K`. Under binary relevance this is Precision@K: `TP / (TP + FP)`. For `[1, 0, 1, 0, 1]`, Relevance@5 is `3 / 5 = 0.6`. Document order does not affect this metric.

In [ ]:
relevance = results["relevance_at_5"]
{
    "score": relevance.score,
    "label": relevance.label,
    "requested_k": relevance.details["requested_k"],
    "effective_k": relevance.details["effective_k"],
    "relevant_count": relevance.details["relevant_count"],
    "documents": relevance.details["documents"],
}

## 9. Hit Rate@K

Hit Rate answers: **Did retrieval find at least one useful result?** It is `1.0` when any relevant document occurs in the top effective K and `0.0` otherwise. Thus `[0, 0, 1, 0, 0] → 1.0`, while `[0, 0, 0, 0, 0] → 0.0`.

In [ ]:
hit_rate = results["hit_rate_at_5"]
{"score": hit_rate.score, "label": hit_rate.label, **hit_rate.details}

## 10. MRR@K

For one `EvaluationCase`, the score is reciprocal rank: `RR = 1 / rank of first relevant document`. Examples: `[1, 0, 1] → 1.0`, `[0, 0, 1] → 1/3`, and `[0, 0, 0] → 0.0`. The evaluator uses the familiar MRR name, but dataset-level MRR is the mean of these per-case reciprocal-rank scores.

In [ ]:
mrr = results["mrr_at_5"]
{
    "score": mrr.score,
    "label": mrr.label,
    "first_relevant_rank": mrr.details["first_relevant_rank"],
}

## 11. nDCG@K

nDCG cares about rank position. `[1, 1, 0, 0, 0]` is better ordered than `[0, 0, 1, 1, 0]`, although both contain two relevant documents. `DCG = sum(relevance_i / log2(rank_i + 1))`; IDCG is DCG after ideally sorting the same relevance values; `nDCG = DCG / IDCG`.

In [ ]:
ndcg = results["ndcg_at_5"]
{
    "score": ndcg.score,
    "label": ndcg.label,
    "relevance_scores": ndcg.details["relevance_scores"],
    "dcg": ndcg.details["dcg"],
    "idcg": ndcg.details["idcg"],
}

## 12. Select only the metrics you need

`metrics=[...]` filters the evaluators already configured on the framework. Only selected metrics are returned, and unselected Hit Rate/MRR make no call. The selected retrieval metrics still share one relevance judgment.

In [ ]:
framework.metrics
selected_results = framework.evaluate(
    case,
    metrics=["relevance_at_5", "ndcg_at_5"],
)
selected_results.keys()

## 13. Different K values

When all configured metrics are selected, documents are judged once through the deepest required effective K. Selecting only `relevance_at_3` judges only the top three; the unselected nDCG@10 does not deepen the call.

In [ ]:
mixed_k_framework = EvaluationFramework(
    judge=judge,
    evaluators=[
        RelevanceAtKEvaluator(k=3),
        HitRateAtKEvaluator(k=5),
        MRRAtKEvaluator(k=5),
        NDCGAtKEvaluator(k=10),
    ],
)
top_three_only = mixed_k_framework.evaluate(
    case, metrics=["relevance_at_3"]
)

## 14. Effective-K behavior

`effective_k = min(requested_k, number_of_retrieved_documents)`. With `k=5` and two returned documents, Relevance@5 divides by two—not five. An empty list returns `score=None`, `label="not_applicable"`, and makes no relevance judge call.

In [ ]:
short_case = EvaluationCase(
    input="How do I reset my password?",
    retrieved_documents=case.retrieved_documents[:2],
)
short_result = framework.evaluate(
    short_case, metrics=["relevance_at_5"]
)["relevance_at_5"]
short_result.details["effective_k"]  # 2

empty_case = EvaluationCase(input=case.input, retrieved_documents=[])
empty_result = framework.evaluate(
    empty_case, metrics=["relevance_at_5"]
)["relevance_at_5"]
(empty_result.score, empty_result.label)  # (None, "not_applicable")

## 15. Structured retrieved documents

Documents can be plain strings or mappings. Only the selected text field is sent to the judge. `document_id`, retrieval `score`, and `metadata` remain diagnostics. Configure `document_text_key="body"` when mappings do not use `text`.

In [ ]:
plain_documents = [
    "Open the recovery page.",
    "Check the verified email account.",
]
structured_documents = [
    {
        "text": "Open the recovery page.",
        "document_id": "d1",
        "score": 0.91,
        "metadata": {"source": "kb"},
    }
]
body_evaluator = RelevanceAtKEvaluator(k=2, document_text_key="body")

## 16. Verbose diagnostics

`verbose=True` additionally exposes judged document text and reasons. It never changes relevance judgments or metric scores, and compact mode avoids duplicating large document payloads.

In [ ]:
verbose_framework = EvaluationFramework(
    judge=judge,
    evaluators=[RelevanceAtKEvaluator(k=5, verbose=True)],
)
verbose_documents = verbose_framework.evaluate(case)["relevance_at_5"].details["documents"]

## 17. Async usage

Jupyter supports top-level `await`. The complete shared relevance call consumes one slot from the framework's shared judge-call concurrency limit.

In [ ]:
async_results = await framework.a_evaluate(
    case,
    metrics=["relevance_at_5", "ndcg_at_5"],
    max_concurrency=4,
)

## 18. `evaluate_many()`

Each independent retrieval case gets its own shared relevance judgment. Sync and async bulk methods preserve case order; async enforces one shared concurrency limit across all cases.

In [ ]:
case_a = case
case_b = EvaluationCase(
    case_id="retrieval-002",
    input="Where can I find billing invoices?",
    retrieved_documents=case.retrieved_documents,
)
cases = [case_a, case_b]
many_results = framework.evaluate_many(
    cases, metrics=["relevance_at_5", "hit_rate_at_5"]
)
async_many_results = await framework.a_evaluate_many(
    cases,
    metrics=["relevance_at_5", "ndcg_at_5"],
    max_concurrency=4,
)

## 19. Context quality metrics

These metrics answer different questions:

- **Relevance@K**: how many retrieved documents are relevant?
- **Contextual Relevancy**: how much meaningful information inside retrieval is useful?
- **Contextual Precision@K**: are relevant documents ranked ahead of irrelevant ones?
- **Contextual Recall**: how much useful authoritative reference information was retrieved?

Contextual Recall requires `context` to be authoritative/gold reference information. It cannot be computed from `input + retrieved_documents` alone.

In [ ]:
context_case = EvaluationCase(
    case_id="retrieval-context-001",
    input="How do I securely reset my password?",
    context=(
        "Password reset requires a verified-email link. The link expires "
        "after 15 minutes. Users may request another link."
    ),
    retrieved_documents=case.retrieved_documents,
)
context_framework = EvaluationFramework(
    judge=judge,
    evaluators=[
        RelevanceAtKEvaluator(k=5),
        ContextualPrecisionAtKEvaluator(k=5),
        ContextualRelevancyEvaluator(verbose=True),
        ContextualRecallEvaluator(verbose=True),
    ],
)
context_results = context_framework.evaluate(context_case)

In [ ]:
{
    name: {
        "score": context_results[name].score,
        "label": context_results[name].label,
    }
    for name in (
        "relevance_at_5",
        "contextual_precision_at_5",
        "contextual_relevancy",
        "contextual_recall",
    )
}

The document metrics and Contextual Precision share **one** relevance call. Contextual Relevancy uses one separate content-unit call, and Contextual Recall uses one separate reference-capture call. Running all seven retrieval/context metrics therefore makes three semantic calls total—not one call per document or item.

## 20. Phoenix integration

Phoenix is used for tracing, observability, and result persistence. Retrieval and context prompts and scoring semantics belong to `idp-eval`; the framework does not use Phoenix built-in metric semantics.

## 21. Close resources

In [ ]:
judge.close()